# CodonFM SAE — Gene-Level Enrichment (GSEA)

The quickstart and codon analysis notebooks examine features at the **codon level** — which codons a feature promotes, what amino acids it responds to, whether it tracks usage bias. But many biological patterns are best understood at the **gene level**: a feature that fires on all ribosomal protein genes, or one that lights up specifically on olfactory receptors.

This notebook runs **Gene Set Enrichment Analysis (GSEA)** on each SAE feature. The idea is simple:
1. For each feature, rank all genes by how strongly the feature activates on them.
2. Test whether the top-ranked genes are enriched for known biological annotations.

We test against five annotation databases, each capturing a different axis of biology:

| Database | What it captures |
|---|---|
| **GO Biological Process** | Pathways and processes (e.g., "translation", "apoptosis") |
| **GO Molecular Function** | Biochemical activity (e.g., "kinase activity", "DNA binding") |
| **GO Cellular Component** | Subcellular location (e.g., "ribosome", "mitochondrial matrix") |
| **InterPro Domains** | Protein domain families (e.g., "immunoglobulin-like fold") |
| **Pfam Domains** | Conserved protein domain families |

Beyond GSEA, we also run two lighter-weight analyses:
- **Gene family detection**: Do the top genes for a feature share a naming prefix (e.g., `OR` for olfactory receptors, `RPS` for ribosomal proteins)?
- **pLI scores** (optional): Are the top genes evolutionarily constrained? pLI (probability of loss-of-function intolerance) from gnomAD measures how much evolutionary pressure there is to keep a gene intact. Features that activate on high-pLI genes may be tracking essential cellular functions.

## Setup

In [ ]:
# Config
SAE_CHECKPOINT = "../outputs/1b_layer16/checkpoints/checkpoint_final.pt"
MODEL_PATH = "../../../../../../../checkpoints/NV-CodonFM-Encodon-TE-Cdwt-1B-v1"
CSV_PATH = "../../../../../../../codonfm/data/codonfm_ref_only.csv"
LAYER = 16
CONTEXT_LENGTH = 2048
BATCH_SIZE = 8
DEVICE = "cuda"
NUM_SEQUENCES = 5000
N_WORKERS = 8

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch


_REPO_ROOT = Path("..").resolve().parent.parent.parent.parent.parent
_CODONFM_TE_DIR = _REPO_ROOT / "recipes" / "codonfm_ptl_te"
sys.path.insert(0, str(_CODONFM_TE_DIR))
sys.path.insert(0, str(Path("..").resolve()))

from codonfm_sae.data import read_codon_csv
from sae.architectures import TopKSAE
from sae.utils import set_seed
from src.data.preprocess.codon_sequence import process_item
from src.inference.encodon import EncodonInference


set_seed(42)
device = DEVICE if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Load SAE & Model

In [ ]:
ckpt = torch.load(SAE_CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = ckpt["model_state_dict"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.removeprefix("module."): v for k, v in state_dict.items()}

input_dim = ckpt.get("input_dim") or state_dict["encoder.weight"].shape[1]
hidden_dim = ckpt.get("hidden_dim") or state_dict["encoder.weight"].shape[0]
model_config = ckpt.get("model_config", {})
top_k = model_config.get("top_k")

sae = TopKSAE(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    top_k=top_k,
    normalize_input=model_config.get("normalize_input", False),
)
sae.load_state_dict(state_dict)
sae = sae.eval().to(device)

print(f"SAE: {input_dim} -> {hidden_dim:,} features (top-{top_k})")

In [ ]:
inference = EncodonInference(
    model_path=MODEL_PATH,
    task_type="embedding_prediction",
    use_transformer_engine=True,
)
inference.configure_model()
inference.model.to(device).eval()

num_layers = len(inference.model.model.layers)
target_layer = LAYER if LAYER >= 0 else num_layers + LAYER
print(f"Encodon: {num_layers} layers, target layer {target_layer}")

## Load Data and Extract Gene Names

GSEA requires gene-level labels. We load the CSV and filter to sequences that have a `gene` column populated. Each sequence maps to one gene, but a gene may appear in multiple sequences (e.g., different transcripts or species). We'll collapse across sequences later by taking the max activation per gene.

In [ ]:
records = read_codon_csv(CSV_PATH, max_sequences=NUM_SEQUENCES, max_codons=CONTEXT_LENGTH - 2)

gene_names = []
valid_records = []
for rec in records:
    gene = rec.metadata.get("gene")
    if gene and str(gene).strip():
        gene_names.append(str(gene).strip())
        valid_records.append(rec)

sequences = [r.sequence for r in valid_records]
print(f"{len(sequences)} sequences with gene names ({len(set(gene_names))} unique genes)")

## Compute Per-Gene Activations

We need to go from per-codon SAE activations to a single score per (gene, feature) pair. The aggregation pipeline is:

1. **Per codon**: Run each sequence through the model + SAE to get activations at every codon position.
2. **Per sequence**: Take the **max** activation across all codons in that sequence, for each feature. This captures whether the feature fires *anywhere* in the sequence.
3. **Per gene**: If a gene appears in multiple sequences, take the **max** across sequences. This gives us the strongest signal the feature has for that gene.

The result is a dictionary: `feature_idx -> gene_name -> activation_score`.

In [ ]:
from tqdm import tqdm


n_features = sae.hidden_dim
n_sequences = len(sequences)

# Phase 1: Compute per-sequence max activations
# Stream one batch at a time to avoid OOM
seq_max_acts = np.zeros((n_sequences, n_features), dtype=np.float32)

with torch.no_grad():
    for i in tqdm(range(0, n_sequences, BATCH_SIZE), desc="Extracting activations"):
        batch_seqs = sequences[i : i + BATCH_SIZE]
        items = [process_item(s, context_length=CONTEXT_LENGTH, tokenizer=inference.tokenizer) for s in batch_seqs]
        batch = {
            "input_ids": torch.tensor(np.stack([it["input_ids"] for it in items])).to(device),
            "attention_mask": torch.tensor(np.stack([it["attention_mask"] for it in items])).to(device),
        }
        out = inference.model(batch, return_hidden_states=True)
        hidden = out.all_hidden_states[LAYER]

        for j, it in enumerate(items):
            seq_len = it["attention_mask"].sum()
            emb = hidden[j, 1 : seq_len - 1, :].float()  # Strip CLS/SEP
            codes = sae.encode(emb)  # [n_codons, n_features]
            seq_max_acts[i + j] = codes.max(dim=0).values.cpu().numpy()

        del out, hidden, batch
        torch.cuda.empty_cache()

print(f"Computed max activations: {seq_max_acts.shape}")

In [ ]:
# Phase 2: Collapse to per-gene max
df = pd.DataFrame(seq_max_acts)
df["gene"] = gene_names
gene_max = df.groupby("gene").max()  # (n_genes, n_features)

# Convert to the dict format expected by run_gene_enrichment:
# feature_idx -> gene_name -> score
gene_activations = {}
for feat_idx in range(n_features):
    col = gene_max[feat_idx]
    nonzero = col[col > 0]
    if len(nonzero) > 0:
        gene_activations[feat_idx] = nonzero.to_dict()

print(f"{len(gene_activations)} features with non-zero gene activations")
print(f"{len(gene_max)} unique genes in activation matrix")

## Run GSEA

For each feature, we run `gseapy.prerank()` against all five annotation databases. This uses the pre-ranked gene list (sorted by activation strength) and tests whether genes belonging to each annotation term are concentrated at the top of the ranking.

We use an FDR (false discovery rate) threshold of 0.05 — a feature is considered "enriched" if at least one term from any database passes this threshold. The `min_size=5` parameter ensures we only test terms with at least 5 genes, avoiding spurious hits from tiny gene sets.

This step is parallelized across features since each GSEA run is independent.

In [ ]:
from codonfm_sae.eval.gene_enrichment import ANNOTATION_DATABASES, run_gene_enrichment


report = run_gene_enrichment(
    gene_activations,
    databases=ANNOTATION_DATABASES,
    fdr_threshold=0.05,
    n_workers=N_WORKERS,
)

print(f"Enriched: {report.n_features_with_enrichment}/{report.n_features_total} ({report.frac_enriched:.1%})")
for db, stats in report.per_database_stats.items():
    print(f"  {db}: {stats['n_enriched']} features, {stats['n_unique_terms']} terms")

## Explore Results

Each enriched feature gets a "best" label — the annotation term with the lowest FDR across all databases. Features enriched for GO Biological Process terms like "translation" or "immune response" are strong evidence that the SAE has learned biologically meaningful decompositions.

Let's look at the top 10 most significantly enriched features.

In [ ]:
# Top enriched features by FDR
sorted_features = sorted(
    report.per_feature,
    key=lambda x: x.overall_best.fdr if x.overall_best else 1.0,
)

print("Top 10 most significantly enriched features:")
print(f"{'Feature':>8}  {'FDR':>8}  {'NES':>6}  {'Database':<30}  {'Term'}")
print("-" * 90)
for fl in sorted_features[:10]:
    if fl.overall_best:
        b = fl.overall_best
        print(f"{fl.feature_idx:>8}  {b.fdr:>8.4f}  {b.enrichment_score:>6.2f}  {b.database:<30}  {b.term_name}")

In [ ]:
# Show all significant terms for a single feature
if sorted_features and sorted_features[0].overall_best:
    example_feat = sorted_features[0]
    print(f"\nAll significant terms for feature {example_feat.feature_idx}:")
    print(f"{'Database':<30}  {'FDR':>8}  {'NES':>6}  {'Term'}")
    print("-" * 90)
    for er in sorted(example_feat.all_significant, key=lambda x: x.fdr):
        print(f"{er.database:<30}  {er.fdr:>8.4f}  {er.enrichment_score:>6.2f}  {er.term_name}")

## Gene Family Detection

A simpler heuristic that complements GSEA: look at the top-K genes for each feature and check if they share a naming prefix. Gene families often use systematic prefixes — `OR` for olfactory receptors, `RPS`/`RPL` for ribosomal proteins, `HLA` for MHC genes, `KRT` for keratins, etc.

A feature where 8 out of 10 top genes start with `OR` is almost certainly tracking olfactory receptor genes, even if GSEA didn't find a significant hit (perhaps because the gene set databases don't have an explicit "olfactory receptor" term).

In [ ]:
from codonfm_sae.eval.gene_enrichment import detect_gene_families


families = detect_gene_families(gene_activations)
print(f"{len(families)} features with dominant gene family")

print(f"\n{'Feature':>8}  {'Family'}")
print("-" * 40)
for feat, label in list(families.items())[:15]:
    print(f"{feat:>8}  {label}")

## pLI Scores (Optional)

**pLI** (probability of Loss-of-function Intolerance) is a measure from the gnomAD project that quantifies how much evolutionary selection pressure acts against loss-of-function mutations in a gene. A pLI score close to 1.0 means the gene is highly constrained — losing one copy is likely lethal or strongly deleterious.

By computing the mean pLI of a feature's top-activating genes, we can ask: **does this feature preferentially fire on essential genes?** Features with high mean pLI may be tracking conserved regulatory patterns or housekeeping functions.

This requires the gnomAD constraint file (`gnomad.v2.1.1.lof_metrics.by_gene.txt.bgz`), which is not included in this repo. Uncomment the cell below if you have it available.

In [ ]:
# Uncomment and update the path if you have the gnomAD pLI file:
# PLI_PATH = "./datasets/gnomad.v2.1.1.lof_metrics.by_gene.txt.bgz"
# pli_scores = load_pli_scores(PLI_PATH)
# print(f"Loaded pLI scores for {len(pli_scores)} genes")
#
# feature_pli = compute_feature_pli(gene_activations, pli_scores)
# print(f"{len(feature_pli)} features with pLI metrics")
#
# # Show features with highest mean pLI (most constrained)
# top_pli = sorted(feature_pli.items(), key=lambda x: x[1]["mean_pli"], reverse=True)[:10]
# print(f"\n{'Feature':>8}  {'Mean pLI':>8}  {'Frac constrained':>16}  {'Max pLI':>8}")
# print("-" * 50)
# for feat, metrics in top_pli:
#     print(f"{feat:>8}  {metrics['mean_pli']:>8.3f}  {metrics['frac_constrained']:>16.3f}  {metrics['max_pli']:>8.3f}")

print("pLI analysis requires gnomAD constraint file. See docstring above.")

## Save Results

Save the enrichment report as JSON for downstream use (e.g., enriching the dashboard atlas, providing GSEA context to auto-interp prompts).

In [ ]:
from dataclasses import asdict


output_dir = Path("../outputs/1b_layer16/gene_enrichment")
output_dir.mkdir(parents=True, exist_ok=True)


# Save report JSON
def _enrichment_result_to_dict(er):
    if er is None:
        return None
    return asdict(er)


report_data = {
    "databases_used": report.databases_used,
    "n_features_with_enrichment": report.n_features_with_enrichment,
    "n_features_total": report.n_features_total,
    "frac_enriched": report.frac_enriched,
    "per_database_stats": report.per_database_stats,
    "significance_threshold": report.significance_threshold,
    "per_feature": [
        {
            "feature_idx": fl.feature_idx,
            "overall_best": _enrichment_result_to_dict(fl.overall_best),
            "go_slim_term": fl.go_slim_term,
            "go_slim_name": fl.go_slim_name,
            "best_per_database": {db: _enrichment_result_to_dict(er) for db, er in fl.best_per_database.items()},
            "n_significant": len(fl.all_significant),
        }
        for fl in report.per_feature
    ],
}

report_path = output_dir / "gene_enrichment_report.json"
with open(report_path, "w") as f:
    json.dump(report_data, f, indent=2)

print(f"Saved report to {report_path}")
print(f"  {report.n_features_with_enrichment} features enriched across {len(report.databases_used)} databases")

## Next Steps

- **04_dashboard.ipynb** — Export feature atlas + examples for the interactive dashboard, incorporating GSEA labels
- **05_auto_interp.ipynb** — Use an LLM to generate natural-language descriptions, with GSEA context as input
- Run the standalone script for production workloads:
  ```bash
  python scripts/eval_gene_enrichment.py \
      --checkpoint outputs/1b_layer16/checkpoints/checkpoint_final.pt \
      --model-path $MODEL_PATH --layer 16 \
      --csv-path $CSV_PATH --n-workers 8 \
      --output-dir outputs/1b_layer16/gene_enrichment
  ```